In [ ]:
import os
import numpy as np
from tqdm import tqdm
from PIL import Image
from scipy.ndimage import binary_dilation
import cv2

def calculate_metrics(folder1, folder2):
    """Calculate IoU, Dice Coefficient, F1-score, and Pixel Accuracy for corresponding segmentation images in two folders."""
    img_files = [f for f in os.listdir(folder1) if f.endswith(('.png', '.jpg', '.jpeg')) and f in os.listdir(folder2)]
    results = []
    
    for img_name in tqdm(img_files, desc="Calculating Metrics"):
        img1_path = os.path.join(folder1, img_name)
        img2_path = os.path.join(folder2, img_name)
        
        img1 = np.array(Image.open(img1_path).convert("L")) > 0  # Convert to binary mask
        img2 = np.array(Image.open(img2_path).convert("L")) > 0  # Convert to binary mask
        
        intersection = np.logical_and(img1, img2).sum()
        union = np.logical_or(img1, img2).sum()
        total_pixels = img1.size
        
        iou = intersection / union if union > 0 else 0
        dice = (2 * intersection) / (img1.sum() + img2.sum()) if (img1.sum() + img2.sum()) > 0 else 0
        f1 = 2 * intersection / (img1.sum() + img2.sum()) if (img1.sum() + img2.sum()) > 0 else 0
        pixel_acc = (img1 == img2).sum() / total_pixels
        
        results.append({
            "image": img_name,
            "IoU": iou,
            "Dice": dice,
            "F1-score": f1,
            "Pixel Accuracy": pixel_acc
        })
    
    return results


def extract_edges(img_path):
    """Extract edges using Canny edge detection and ignore the image boundary (first & last row/column)."""
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # Read as grayscale
    edges = cv2.Canny(img, 100, 200) > 0  # Apply Canny edge detection (binary mask)

    # Remove the first and last rows/columns (ignoring image boundaries)
    edges[0, :] = False  # First row
    edges[-1, :] = False  # Last row
    edges[:, 0] = False  # First column
    edges[:, -1] = False  # Last column
    
    return edges

def calculate_edge_overlap(folder1, folder2):
    """Calculate edge overlap while ignoring the first & last row/column (image boundaries)."""
    img_files = [f for f in os.listdir(folder1) if f.endswith(('.png', '.jpg', '.jpeg')) and f in os.listdir(folder2)]
    results = []
    
    for img_name in tqdm(img_files, desc="Calculating Edge Overlap"):
        img1_path = os.path.join(folder1, img_name)
        img2_path = os.path.join(folder2, img_name)
        
        img1 = extract_edges(img1_path)  # Extract edges for image 1
        img2 = extract_edges(img2_path)  # Extract edges for image 2
        
        same_pixels = np.logical_and(img1, img2).sum()  # Exact edge overlap
        
        # Define neighborhood structures
        struct_4 = np.array([[0,1,0], [1,1,1], [0,1,0]], dtype=bool)  # 4-neighborhood
        struct_8 = np.array([[1,1,1], [1,1,1], [1,1,1]], dtype=bool)  # 8-neighborhood
        struct_24 = np.ones((5,5), dtype=bool)  # 24-neighborhood (5x5 window)
        struct_48 = np.ones((7,7), dtype=bool)  # 48-neighborhood (7x7 window)
        
        img1_4_neighbor = binary_dilation(img1, structure=struct_4)
        img1_8_neighbor = binary_dilation(img1, structure=struct_8)
        img1_24_neighbor = binary_dilation(img1, structure=struct_24)
        struct_48_neighbor = binary_dilation(img1, structure=struct_48)
        
        within_4_neighbor = np.logical_and(img1_4_neighbor, img2).sum()
        within_8_neighbor = np.logical_and(img1_8_neighbor, img2).sum()
        within_24_neighbor = np.logical_and(img1_24_neighbor, img2).sum()
        within_48_neighbor = np.logical_and(struct_48_neighbor, img2).sum()
        
        results.append({
            "image": img_name,
            "mSCR": 2*same_pixels / (max(1, img1.sum()) + max(1, img2.sum())),
            "mSCR-N4": 2*within_4_neighbor / (max(1, img1.sum()) + max(1, img2.sum())),
            "mSCR-N8": 2*within_8_neighbor / (max(1, img1.sum()) + max(1, img2.sum())),
            "mSCR-N24": 2*within_24_neighbor / (max(1, img1.sum()) + max(1, img2.sum())),
            "mSCR-N48": 2*within_48_neighbor / (max(1, img1.sum()) + max(1, img2.sum()))
        })
    
    return results

In [ ]:
detect_folder_list = [  '/home/weiwang/ResearchProjects/mmsegmentation/test_set/PSP_result/',
                        '/home/weiwang/ResearchProjects/mmsegmentation/test_set/PSA_result/',
                        '/home/weiwang/ResearchProjects/mmsegmentation/test_set/OCR_result/',
                        '/home/weiwang/ResearchProjects/mmsegmentation/test_set/FCN_result/',
                        '/home/weiwang/ResearchProjects/mmsegmentation/test_set/ENCNet_result/',
                        '/home/weiwang/ResearchProjects/mmsegmentation/test_set/Deeplabv3_result/',
                        '/home/weiwang/ResearchProjects/mmsegmentation/test_set/DaNet_result/',
                        '/home/weiwang/ResearchProjects/mmsegmentation/test_set/CCNet_result/',
                        '/home/weiwang/ResearchProjects/mmsegmentation/test_set/ANN_result/'] 
detect_key_list = ['PSP', 'PSA', 'OCR', 'FCN', 'ENCNet', 'Deeplabv3', 'DANet', 'CCNet', 'ANN']
GT_folder = '/home/weiwang/ResearchProjects/mmsegmentation/test_set/GT_label/'
res_dict_Stat, res_dict_Edge = {}, {}
for i in range(len(detect_folder_list)):
    res_dict_Stat[detect_key_list[i]] = calculate_metrics(detect_folder_list[i], GT_folder)
    


Calculating Edge Overlap: 100%|██████████| 3104/3104 [00:22<00:00, 136.80it/s]


In [15]:
for i in range(len(detect_folder_list)):
    res_dict_Edge[detect_key_list[i]] = calculate_edge_overlap(detect_folder_list[i], GT_folder)


Calculating Edge Overlap: 100%|██████████| 3104/3104 [01:06<00:00, 46.74it/s]


In [ ]:
# create the python script to generate the table, that the first column is the model name, and the rest columns are the mean value of the metrics
# the metrics are IoU, Dice, F1-score, Pixel Accuracy, Same Pixel Ratio, Within 4-Neighbor Ratio, Within 8-Neighbor Ratio
# the table is saved in the file 'table.txt'

table_file = open('table.txt', 'w')
table_file.write('Model\tIoU\tDice\tF1-score\tPixel Accuracy\tSame Pixel Ratio\tWithin 4-Neighbor Ratio\tWithin 8-Neighbor Ratio\n')
for j in range(len(detect_key_list)):
    i = len(detect_key_list) - j - 1
    model_name = detect_key_list[i]
    res_stat = res_dict_Stat[model_name]
    res_edge = res_dict_Edge[model_name]
    IoU = sum([res['IoU'] for res in res_stat]) / len(res_stat)
    Dice = sum([res['Dice'] for res in res_stat]) / len(res_stat)
    F1_score = sum([res['F1-score'] for res in res_stat]) / len(res_stat)
    Pixel_Accuracy = sum([res['Pixel Accuracy'] for res in res_stat]) / len(res_stat)
    Same_Pixel_Ratio = sum([res['mSCR'] for res in res_edge]) / len(res_edge)
    Within_4_Neighbor_Ratio = sum([res['mSCR-N4'] for res in res_edge]) / len(res_edge)
    Within_8_Neighbor_Ratio = sum([res['mSCR-N8'] for res in res_edge]) / len(res_edge)
    Within_24_Neighbor_Ratio = sum([res['mSCR-N24'] for res in res_edge]) / len(res_edge)
    Within_48_Neighbor_Ratio = sum([res['mSCR-N48'] for res in res_edge]) / len(res_edge)
    table_file.write(f'{model_name}\t{IoU:.4f}\t{Dice:.4f}\t{F1_score:.4f}\t{Pixel_Accuracy:.4f}\t{Same_Pixel_Ratio:.4f}\t{Within_4_Neighbor_Ratio:.4f}\t{Within_8_Neighbor_Ratio:.4f}\t{Within_24_Neighbor_Ratio:.4f}\t{Within_48_Neighbor_Ratio:.4f}\n')

    # table_file.write(f'{model_name}\t{IoU:.4f}\t{Dice:.4f}\t{F1_score:.4f}\t{Pixel_Accuracy:.4f}\t{Same_Pixel_Ratio:.4f}\t{Within_4_Neighbor_Ratio:.4f}\t{Within_8_Neighbor_Ratio:.4f}\n')

table_file.close()
print('The table is saved in the file "table.txt"')

The table is saved in the file "table.txt"
